# Week 5 Problem Set

## Homework

In [2]:
%load_ext nb_mypy
%nb_mypy Off

Version 1.0.6


In [3]:
from typing import TypeAlias
from typing import Optional, Any, Iterator
from __future__ import annotations

Number: TypeAlias = int | float
NumberList: TypeAlias = list[int|float] 

**HW1.** Modify the class `TurtleWorld` to include the following attribute and methods:
- `movement_queue` which is an attribute of the type `Queue` to store the movement list. **Each entry of this queue is an object which has the name of the turtle (e.g. "t1") and the movement list (e.g. "ulrr"). The choice of this object is left to you, e.g. it could be a tuple (`("t1", "ulrr")`) or a dictionary with the same information or your own custom class.
- `add_movement(turtle, movement)` which adds turtle movement to the queue `movement_queue` to be run later. The argument `turtle` is a string containing the turtle's name. The argument `movement` is another string for the movement. For example, value for `turtle` can be something like `'t1'` while the value for the `movement` can be something like `'uullrrdd'`.
- `run()` which executes all the movements in the queue.

In [4]:
import math

class Coordinate:
    
    def __init__(self, x:Number=0, y:Number=0) -> None:
        self.x = x
        self.y = y
        
    @property
    def distance(self) -> float:
        return math.sqrt(self.x * self.x + self.y * self.y)
    
    def __str__(self) -> str:
        return f"({self.x}, {self.y})"

In [5]:
# copy and paste your solution for Queue class

class Queue:
    def __init__(self) -> None:
        self.__items: list[Any] = []
        
    @property
    def items(self):
        return self.__items
    
    @items.setter
    def items(self, value):
        self.__items = value
    
    def enqueue(self, item: Any) -> None:
        self.__items.append(item)
    
    def dequeue(self) -> Any:
        if self.is_empty:
            return None
        
        return self.__items.pop(0)
    
    def peek(self) -> Any:
        if self.is_empty:
            return None
        
        return self.__items[-1]
    
    @property
    def is_empty(self) -> bool:
        return len(self.__items) == 0
    
    @property
    def size(self) -> int:
        return len(self.__items)

In [6]:
# Class definition
class RobotTurtle:
    # Attributes:
    def __init__(self, name: str, speed: int=1) -> None:
        assert isinstance(name, str) and name != ""
        assert isinstance(speed, int) and speed > 0
        self._name: str = name
        self._speed: int = speed
        self._pos: Coordinate = Coordinate(0, 0)
        
    # property getter
    @property
    def name(self) -> str:
        return self._name
    
    # property setter
    @name.setter
    def name(self, value: str) -> None:
        if isinstance(value, str) and value != "":
            self._name = value
            
    # property getter
    @property
    def speed(self) -> int:
        return self._speed
    
    # property setter
    @speed.setter
    def speed(self, value: int) -> None:
        if isinstance(value, int) and value > 0:
            self._speed = value

    # property getter
    @property
    def pos(self) -> Coordinate:
        return self._pos
    
    # Methods:
    def move(self, direction: str) -> None:
        update: dict[str, Coordinate] = {'up' : Coordinate(self.pos.x, self.pos.y + self.speed),
                                        'down' : Coordinate(self.pos.x, self.pos.y - self.speed),
                                        'left' : Coordinate(self.pos.x - self.speed, self.pos.y),
                                        'right' : Coordinate(self.pos.x + self.speed, self.pos.y)}
        self._pos = update[direction]

        
    def tell_name(self) -> None:
        print(f"My name is {self.name}")


In [7]:
class TurtleWorld:
    valid_movements:set[str] = set('udlr')
    movement_map: dict[str, str] = {'u': 'up', 'd': 'down', 'l': 'left', 'r': 'right'}
    
    def __init__(self) -> None:
        self.turtles: dict[str, RobotTurtle] = {}
        # add a code to create a Queue for the movement
        self.movement_queue = Queue()
        
    def add_movement(self, turtle: str, movement: str) -> None:
        toEnqueue = (turtle, movement)
        self.movement_queue.enqueue(toEnqueue)
    
    def run(self) -> None:
        while not self.movement_queue.is_empty:
            # destructure the tuple to extract data
            robotName, robotMove = self.movement_queue.dequeue()
            # execute the move
            self.move_turtle(robotName, robotMove)
        
    def move_turtle(self, name: str, movement: str) -> None:
        # self[name] -> target the RobotTurtle
        # read from movement -> each character we check with the movement_map
        theRobot = self.turtles[name]
        for move in movement:
            if move not in self.valid_movements:
                continue
            direction = self.movement_map[move]
            # move depends on speed
            theRobot.move(direction)
    
    def add_turtle(self, name: str, speed: int) -> None:
        self.turtles[name] = RobotTurtle(name, speed)
        
    def remove_turtle(self, name: str) -> None:
        del self.turtles[name]
        
    def list_turtles(self) -> list[str]:
        return sorted( self.turtles.keys() )

In [8]:
world: TurtleWorld = TurtleWorld()
assert isinstance(world.movement_queue, Queue)

world.add_turtle('t1', 1)
world.add_turtle('t2', 2)
world.add_movement('t1', 'ur')
world.add_movement('t2', 'urz')
assert str(world.turtles['t1'].pos) == '(0, 0)'
assert str(world.turtles['t2'].pos) == '(0, 0)'
assert world.movement_queue.size == 2

world.run()
assert str(world.turtles['t1'].pos) == '(1, 1)'
assert str(world.turtles['t2'].pos) == '(2, 2)'

world.add_movement('t1', 'ur')
world.add_movement('t2', 'urz')

world.run()
assert str(world.turtles['t1'].pos) == '(2, 2)'
assert str(world.turtles['t2'].pos) == '(4, 4)'



In [9]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW2.** Implement a radix sorting machine. A radix sort for base 10 integers is a *mechanical* sorting technique that utilizes a collection of bins:
- one main bin 
- 10 digit-bins

Each bin acts like a *queue* and maintains its values in the order that they arrive. The algorithm works as follows:
- it begins by placing each number in the main bin. 
- Then it considers each value digit by digit. The first value is removed from the main bin and placed in a digit-bin corresponding to the digit being considered. For example, if the ones digit is being considered, 534 will be placed into digit-bin 4 and 667 will placed into digit-bin 7. 
- Once all the values are placed into their corresponding digit-bins, the values are collected from bin 0 to bin 9 and placed back in the main bin (in that order). 
- The process continues with the tens digit, the hundreds, and so on. 
- After the last digit is processed, the main bin will contain the values in ascending order.

Create a class `RadixSort` that takes in a List of Integers during object instantiation. The class should have the following properties:
- `items`: is a List of Integers containing the numbers.

It should also have the following methods:
- `sort()`: which returns the sorted numbers from `items` as an `list` of Integers.
- `max_digit()`: which returns the maximum number of digits of all the numbers in `items`. For example, if the numbers are 101, 3, 1041, this method returns 4 as the result since the maximum digit is four from 1041. 
- `convert_to_str(items)`: which returns items as a list of Strings (instead of Integers). This function should pad the higher digits with 0 when converting an Integer to a String. For example if the maximum digit is 4, the following items are converted as follows. From `[101, 3, 1041]` to `["0101", "0003", "1041"]`.

Hint: Your implementation should make use of the generic `Queue` class, which you created, for the bins.

In [10]:
class RadixSort:
    
    def __init__(self, my_list: list[int]) -> None:
        self.items = my_list
    
    def max_digit(self) -> int:
        max_digits = 0
        
        if len(self.items) == 0: return 0
        
        for number in self.items:
            # convert number to string first
            number = str( abs(number) )
            
            # get the number of digits
            digits = len(number)
            
            # update maximum digits if needed
            if digits > max_digits:
                max_digits = digits
                
        return max_digits
    
    def convert_to_str(self, items: list[int]) -> list[str]:
        str_list = []
        maximum_digit = self.max_digit()
        
        for item in items:
            # convert a number to string first
            item = str( abs(item) )
            
            # add leading 0s
            number_of_0 = maximum_digit - len(item)
            item = "0" * number_of_0 + item
                
            str_list.append(item)
            
        # convert list of int to list of str
        return str_list
    
    def sort(self) -> list[int]:
        # main bin
        main_bin = Queue()
        
        # 10 radix bins
        radix_bins = [Queue() for _ in range(10)]
        
        # find max digit
        maxDigits = self.max_digit()
        
        # convert into the correct format
        # use a local list so self.items keeps its original integers (optional), the self.items does not matter also
        str_items = self.convert_to_str(self.items)
        
        # put all number in array:
        for number in str_items:
            main_bin.enqueue(number)
            
        # we go through each digits inside a number, starting from the last digit
        for digit_pos in range( maxDigits ):
            
            # take each number from the main queue
            # extract the last digit
            # each number here is in the string form
            # to extract: number[-(digit_pos + 1)]
            # e.g: access last digit -> digit_pos = 0 => number[-1]
            # e.g: 2nd last digit -> digit_pos = 1 => number[-2]
            while len(main_bin.items) != 0:
                current_number = main_bin.dequeue()
                extracted_digit = int( current_number[-(digit_pos + 1)] )
                radix_bins[extracted_digit].enqueue(current_number)
            
            # after numbers are put in their correct bins
            # take them out and put back to main queue
            # in an order starting from bin 0
            for radix_bin in radix_bins:
                while len(radix_bin.items) != 0:
                    current_number = radix_bin.dequeue()
                    
                    main_bin.enqueue(current_number)
        
        # the result main_bin will be in the form of string
        # convert all to integers
        main_bin.items = [int(item) for item in main_bin.items]
        
        return main_bin.items
    
        

In [11]:
list1: RadixSort = RadixSort([101, 3, 1041])
assert list1.items == [101,3,1041]
assert list1.max_digit() == 4
assert list1.convert_to_str(list1.items) == ["0101", "0003", "1041"]
ans: list[int] = list1.sort()
print(ans)
assert ans == [3, 101, 1041]
list2: RadixSort = RadixSort([23, 1038, 8, 423, 10, 39, 3901])
assert list2.sort() == [8, 10, 23, 39, 423, 1038, 3901]

[3, 101, 1041]


In [12]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW3**. Create a class `Rectangle` which is a subclass of `GeometricObject`. It has two additional properties:
- `width` with default value of 1.0.
- `height` with default value of 1.0

It also has two computed properties:
- `area`
- `perimeter`

It has one method:
- `print()` that displays the information about the rectangle in the following format: "colour: green and filled: True and area: xx.xx". Use string formating to format the output area to two decimal places. 

In [13]:
class GeometricObject:
    VALID_COLOURS: set[str] = {"red", "green", "blue", "white", "black"}

    def __init__(self, colour: str = "green", filled: bool = True) -> None:
        assert isinstance(colour, str)
        assert isinstance(filled, bool)
        self._colour = colour
        self._filled = filled
    
    @property
    def colour(self):
        return self._colour
    
    @colour.setter
    def colour(self, val):
        if val in self.VALID_COLOURS:
            self._colour = val
        else:
            self._colour = "green"
            
    @property
    def filled(self):
        return self._filled
    
    @filled.setter
    def filled(self, val):
        if isinstance(val, bool):
            self._filled = val
        
    def __str__(self) -> str:
        return f"colour: {self.colour:s} and filled: {str(self.filled):s}"

In [14]:
class Rectangle(GeometricObject):
    def __init__(self, width: float = 1.0, height: float = 1.0) -> None:
        super().__init__()
        self.width = width
        self.height = height

    @property
    def width(self) -> float:
        return self._width

    @width.setter
    def width(self, val: float) -> None:
        if isinstance(val, (int, float)) and val > 0:
            self._width = float(val)

    @property
    def height(self) -> float:
        return self._height

    @height.setter
    def height(self, val: float) -> None:
        if isinstance(val, (int, float)) and val > 0:
            self._height = float(val)

    @property
    def area(self) -> float:
        return self.width * self.height

    @property
    def perimeter(self) -> float:
        return 2 * (self.width + self.height)

    def print(self) -> None:
        print(f"{self} and area: {self.area:.2f}")

In [15]:
r: Rectangle = Rectangle()
assert r.area == 1.0
r.width = 2.0
r.height = 3.0
assert r.area == 6.0
rect: Rectangle = Rectangle(3.0, 2.0)
rect.colour = "red"
rect.filled = True
assert rect.width == 3.0
assert rect.height == 2.0
assert rect.area == 6.0
assert rect.colour == "red"
assert rect.filled == True


In [16]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW4.** Write a class called `EvaluateFraction` that evaluates postfix notation implemented using Stack and Queue data structures. Postfix notation is a way of writing expressions without using parenthesis. For example, the expression `(1+2)*3` would be written as `1 2 + 3 *`. The class `EvaluateFraction` has the following method:
- `input(inp)`: which pushes the input input one at a time. For example, to create a postfix notation `1 2 + 3 *`, we can call this method repetitively, e.g. `e.input('1'); e.input('2'); e.input('+'); e.input('3'); e.input('*')`. Notice that the input is of String data type. 
- `evaluate()`: which returns the output of the expression.
- `get_fraction(inp)`: which takes in an input string and returns a `Fraction` object. 

Postfix notation is evaluated using a Stack. The input streams from `input()` are stored in a Queue. If the output of the Queue is a number, the item is pushed into the stack. If it is an operator, we will apply the operator to the two top most item n the stacks and push the result back into the stack. 

In [17]:
class Stack:
    def __init__(self) -> None:
        self.__items: list[Any] = []
        
    def push(self, item: Any) -> None:
        self.__items.append(item)

    def pop(self) -> Any:
        if self.is_empty:
            return None
        
        return self.__items.pop()

    def peek(self) -> Any:
        if self.is_empty:
            return None
        
        return self.__items[-1]

    @property
    def is_empty(self) -> bool:
        return len(self.__items) == 0

    @property
    def size(self) -> int:
        return len(self.__items)

In [18]:
class Queue:
    def __init__(self) -> None:
        self.in_stack: Stack = Stack()
        self.out_stack: Stack = Stack()
        
    def enqueue(self, item):
        self.in_stack.push(item)
    
    def dequeue(self):
        if self.out_stack.is_empty:
            while not self.in_stack.is_empty:
                self.out_stack.push( self.in_stack.pop() )
            
        return self.out_stack.pop()
    
    def peek(self):
        if self.out_stack.is_empty:
            while not self.in_stack.is_empty:
                self.out_stack.push( self.in_stack.pop() )
            
        return self.out_stack.peek()
    
    @property
    def is_empty(self) -> bool:
        return self.in_stack.is_empty and self.out_stack.is_empty
    
    @property
    def size(self) -> int:
        return self.in_stack.size + self.out_stack.size

In [19]:
def gcd(a: int, b: int) -> int:
    if b == 0:
        return a
    else:
        return gcd(b, a % b)


class Fraction:
    def __init__(self, num: int, den: int) -> None:
        assert isinstance(num, int)
        assert isinstance(den, int)
        self._num = num
        if den == 0:
            self._den = 1
        else:
            self._den = den

    
    @property
    def num(self) -> int:
        return self._num
    
    @num.setter
    def num(self, val: int) -> None:
        self._num = int(val) if isinstance(val, float) else val
        
        
    @property
    def den(self) -> int:
        return self._den
    
    @den.setter
    def den(self, val: int) -> None:
        if not isinstance(val, (int, float)):
            return
        val = int(val)
        self._den = 1 if val == 0 else val
    
    def __str__(self) -> str:
        return f"{self.num}/{self.den}"
    
    
    def simplify(self) -> Fraction:
        common_divisor = gcd(self.den, self.num)
        
        # new numerator and denominator
        new_den = self.den // common_divisor
        new_num = self.num // common_divisor
        
        return Fraction(new_num, new_den)
    
    def __add__(self, other) -> Fraction:
        result_num = self.num * other.den + self.den * other.num
        result_den = self.den * other.den
        
        
        return Fraction(result_num, result_den).simplify()
        
    def __sub__(self, other) -> Fraction:
        result_num = self.num * other.den - self.den * other.num
        result_den = self.den * other.den

        return Fraction(result_num, result_den).simplify()
    
    def __mul__(self, other) -> Fraction:
        result_num = self.num * other.num
        result_den = self.den * other.den

        return Fraction(result_num, result_den).simplify()
    
    def __eq__(self, other) -> bool:
        simplified_self = self.simplify()
        simplified_other = other.simplify()

        return (
            simplified_self.num == simplified_other.num
            and simplified_self.den == simplified_other.den
        )
    
    def __lt__(self, other) -> bool:
        return self.num * other.den < other.num * self.den
    
    def __le__(self, other) -> bool:
        return self < other or self == other
    
    def __gt__(self, other) -> bool:
        return other < self
    
    def __ge__(self, other) -> bool:
        return other <= self
    


In [20]:
class EvaluateFraction:

    operands: str = "0123456789"
    operators: str = "+-*/"
    
    def __init__(self) -> None:
        self.expression: Queue = Queue()
        self.stack: Stack = Stack()
    
    def input(self, item: str) -> None:
        self.expression.enqueue(item)
    
    def evaluate(self) -> Fraction:
        # scan from left to right
        while not self.expression.is_empty:
            # we loop through the queue the queue
            element = self.expression.dequeue()
            
            # if element is an operator
            if element in self.operators:
                # these are in the type of Fraction (class)
                right_frac = self.stack.pop()
                left_frac = self.stack.pop()
                result = self.process_operator(left_frac, right_frac, element)
                
                self.stack.push(result)
            #  if element is a string that contains fraction
            else:
                # convert them to Fraction type first
                element = self.get_fraction(element)
                self.stack.push(element)
                
        return self.stack.pop()
    
    def get_fraction(self, inp: str) -> Fraction:
        parts = inp.split('/')
        numerator = int( parts[0] )
        denominator = int( parts[1] )
        
        return Fraction(numerator, denominator)
    
    def process_operator(self, op1: Fraction, op2: Fraction, op: str) -> Fraction:
        if op == '+':
            resultFraction = op1 + op2
        elif op == '-':
            resultFraction = op1 - op2
        elif op == '*':
            resultFraction = op1 * op2
        elif op == '/':
            resultFraction = op1 / op2
        
        return resultFraction

In [21]:
pe: EvaluateFraction = EvaluateFraction()
pe.input("1/2")
pe.input("2/3")
pe.input("+")
assert pe.evaluate()==Fraction(7, 6)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1/6")
pe.input("-")
assert pe.evaluate()==Fraction(1, 1)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1/6")
pe.input("-")
pe.input("3/4")
pe.input("*")
assert pe.evaluate()==Fraction(3, 4)

In [22]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###

**HW5.** Modify HW2 so that it can work with MixedFraction. Write a class called `EvaluateMixedFraction` as a subclass of `EvaluateFraction`. You need to override the following methods:
- `get_fraction(inp)`: This function should be able to handle string input for MixedFraction such as `1 1/2` or `3/2`. It should return a `MixedFraction` object.
- `evaluate()`: This function should return `MixedFraction` object rather than `Fraction` object. 

In [23]:
class MixedFraction(Fraction):
    def __init__(self, top: int, bot: int, whole: int=0) -> None:
        num = whole * bot + top
        super().__init__(num, bot)

    def get_three_numbers(self) -> tuple[int, int, int]:
        whole = self.num // self.den
        top = self.num % self.den
        bot = self.den
        
        return (top, bot, whole)

    def __str__(self) -> str:
        # destructuring top, bot, whole
        top, bot, whole = self.get_three_numbers()
        if whole == 0:
            return f"{top}/{bot}"
        
        return f"{whole} {top}/{bot}"

In [24]:
class EvaluateMixedFraction(EvaluateFraction):
    def get_fraction(self, inp: str) -> Fraction:
        # if it's mixed fraction
        if " " in inp:
            whole, frac = inp.split(" ")
            whole = int(whole)
            top, bot = frac.split("/")
        # if it's just one fraction
        else:
            whole = 0
            top, bot = inp.split('/')
            
        return MixedFraction(int(top), int(bot), whole)
    
    def evaluate(self) -> MixedFraction:
        # how we evaluateMixedFunction is using the same method as for general fraction
        answer = super().evaluate()
        
        return MixedFraction(answer.num, answer.den)

In [25]:
pe: EvaluateMixedFraction = EvaluateMixedFraction()
pe.input("3/2")
pe.input("1 2/3")
pe.input("+")
out: MixedFraction = pe.evaluate() 
assert out == MixedFraction(1, 6, 3)
assert isinstance(out, MixedFraction)

pe.input("1/2")
pe.input("2/3")
pe.input("+")
pe.input("1 1/8")
pe.input("-")
assert pe.evaluate() == MixedFraction(1, 24)

pe.input("1 1/2")
pe.input("2 2/3")
pe.input("+")
pe.input("1 1/6")
pe.input("-")
pe.input("5/4")
pe.input("*")
assert pe.evaluate() == MixedFraction( 3, 4, 3)

In [26]:
###
### AUTOGRADER TEST - DO NOT REMOVE
###